# Principles of CNNs

This notebook explores convolutional neural networks (CNNs) and compares them with multilayer perceptrons (MLPs). The discussion draws on *Deep Learning* by Ian Goodfellow, Yoshua Bengio, and Aaron Courville.

Convolutional layers encode three useful assumptions about images:

1. **Local connectivity:** Each output at a spatial location depends on a small input neighborhood. A dense layer instead connects each output to every input. Small convolution kernels can substantially reduce parameter storage and computation relative to a dense layer with the same input and output shapes.

2. **Parameter sharing:** The same kernel is used at every spatial location. This allows a feature detector to respond to a pattern wherever it appears, while using fewer independent parameters than a locally connected layer with separate weights at every location.

3. **Translation equivariance:** If $T$ translates an image, an equivariant feature extractor satisfies $f(Tx)=T f(x)$: shifting the input shifts the feature map. Stride-one convolutions and pointwise activations such as ReLU have this property on an infinite grid, or with compatible periodic boundaries. On finite images with zero padding, boundary effects can break exact equivariance; compare locations whose full receptive fields stay inside the image. Striding can also restrict which shifts are equivariant.

Equivariance differs from **invariance**, which would mean $f(Tx)=f(x)$. Neither translation invariance nor equivariance to arbitrary rotations or rescalings is guaranteed for a generic CNN classifier. Data augmentation can encourage robustness to such transformations.


In [ ]:
# As usual, a bit of setup

import IPython
from IPython.display import display, HTML

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import sampler

import torchvision.datasets as dset
import torchvision.transforms as T
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import random

seed = 7
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

# for auto-reloading external modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

## MLP vs. CNN

Part A compares how untrained CNN and MLP feature maps respond when an image moves. Part B compares classification performance on CIFAR-10 using an MLP and a CNN with similar total parameter counts.

Similar parameter counts provide a useful comparison of model size, but do not imply equal expressiveness, computational cost, or training behavior. Record the results you actually observe; a single run does not establish that one architecture always performs better.


Part A downloads MNIST for the translation visualization. Part B downloads CIFAR-10 for training. The first downloads may take a few minutes; subsequent runs can reuse the cached files.

PyTorch provides utilities to download these datasets, preprocess examples, and form minibatches.


You have an option to **use GPU by setting the flag to True below**. It is recommended, but not necessary to use GPU for this assignment. Note that if your computer does not have CUDA enabled, `torch.cuda.is_available()` will return False and this notebook will fallback to CPU mode.

The global variables `dtype` and `device` will control the data types throughout this assignment.

## Colab Users

If you are using Colab, you need to manually switch to a GPU device. You can do this by clicking `Runtime -> Change runtime type` and selecting `GPU` under `Hardware Accelerator`. Note that you have to rerun the cells from the top since the kernel gets restarted upon switching runtimes.

In [ ]:
USE_GPU = True

dtype = torch.float32 # we will be using float throughout this tutorial

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

# Constant to control how frequently we print train loss
print_every = 500

print('using device:', device)

## **Part A. Translation Equivariance**

Run the following cells and use the slider to move the same digit within a larger image. Both networks have randomly initialized weights and remain untrained. Compare how the feature maps move, keeping in mind that zero padding can affect the image boundaries.


In [ ]:
## Some helpers
def torch_to_numpy(tensor):
    tensor = tensor.cpu().detach().numpy()
    return tensor

def preprocess_mnist_data(data):
    # padding tuples: (padding_left, padding_right, padding_top, padding_bottom)
    # data1 = F.pad(data, (0, 28, 0, 28), mode='constant', value=0)
    # data2 = F.pad(data, (28, 0, 0, 28), mode='constant', value=0)
    # data3 = F.pad(data, (0, 28, 28, 0), mode='constant', value=0)
    # data4 = F.pad(data, (28, 0, 28, 0), mode='constant', value=0)
    # data = torch.cat((data1, data2, data3, data4), dim=0)

    padded_data_list = []

    for i in range(0, 28, 4):
        for j in range(0, 28, 4):
            padded_data_list.append(F.pad(data, (i, 28-i, j, 28-j), mode='constant', value=0))

    padded_data = torch.stack(padded_data_list, dim=0)

    return padded_data

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 20, 3, 1, padding=1)
        self.conv2 = nn.Conv2d(20, 40, 3, 1, padding=1)
        self.conv3 = nn.Conv2d(40, 1, 3, 1, padding=1)
        # self.conv4 = nn.Conv2d(40, 1, 3, 1, padding=1)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        # x = F.relu(self.conv4(x))
        return x

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(56*56, 100)
        self.fc2 = nn.Linear(100, 100)
        self.fc3 = nn.Linear(100, 56*56)

    def forward(self, x):
        bs = x.shape[0]
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = x.reshape((bs, 1, 56, 56))

        return x


### Generate the sample data

In [ ]:
mnist_train = dset.MNIST('./deeplearning/datasets', train=True, download=True)
sample_index = 12
mnist_sample = mnist_train[sample_index][0]
mnist_sample = T.functional.pil_to_tensor(mnist_sample)

mnist_sample = preprocess_mnist_data(mnist_sample)
print(mnist_sample.shape)

### Visualize the output of MLP and CNN with different translations

In [ ]:
# !pip3 install ipywidgets

In [ ]:
from ipywidgets import interactive, widgets, Layout

In [ ]:
cnn_model = CNN().to(device)
mlp_model = MLP().to(device)

mnist_sample = mnist_sample.to(device)
# Convert to float32
mnist_sample = mnist_sample.float()

cnn_output = torch_to_numpy(cnn_model(mnist_sample))
mlp_output = torch_to_numpy(mlp_model(mnist_sample))
mnist_sample = torch_to_numpy(mnist_sample)

In [ ]:
# Move the same digit while comparing the two untrained networks.
def update_images(i):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    images = [mnist_sample[i, 0], cnn_output[i, 0], mlp_output[i, 0]]
    titles = ['Input image', 'CNN output', 'MLP output']
    for axis, image, title in zip(axes, images, titles):
        axis.imshow(image)
        axis.set_title(title)
        axis.axis('off')
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()
    plt.close(fig)

ip = interactive(
    update_images,
    i=widgets.IntSlider(min=0, max=len(mnist_sample) - 1, step=1, value=0),
)
display(ip)


**Q.** As the digit moves, how do the CNN and MLP output patterns change? Explain the architectural reason, and state the limitation caused by zero padding.

**A.** 

**Q.** Neither network has been trained. Which of the properties you observed follows from the architecture alone? Does a recognizable feature map guarantee good classification accuracy?

**A.** 


## **Part B. Training a CNN and an MLP on CIFAR-10**

We now train classifiers on [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html). These architectures differ from the feature extractors in Part A. The code reports **validation accuracy** on 1,000 held-out training examples; it does not report test accuracy.


In [ ]:
NUM_TRAIN = 49000

# The torchvision.transforms package provides tools for preprocessing data
# and for performing data augmentation; here we set up a transform to
# preprocess the data by subtracting the mean RGB value and dividing by the
# standard deviation of each RGB value; we've hardcoded the mean and std.
# If we want to add data augmentations, torchvision also offers different
# transformations that we can compose in here, though we would need to be sure
# to have two sets of transformations: one with data augmentation for the
# training loaders, and one without for the test and validation sets.
transform = T.Compose([
                T.ToTensor(),
                T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
            ])

# We set up a Dataset object for each split (train / val / test); Datasets load
# training examples one at a time, so we wrap each Dataset in a DataLoader which
# iterates through the Dataset and forms minibatches. We divide the CIFAR-10
# training set into train and val sets by passing a Sampler object to the
# DataLoader telling how it should sample from the underlying Dataset.
cifar10_train = dset.CIFAR10('./deeplearning/datasets', train=True, download=True,
                             transform=transform)
loader_train = DataLoader(cifar10_train, batch_size=64,
                          sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

cifar10_val = dset.CIFAR10('./deeplearning/datasets', train=True, download=True,
                           transform=transform)
loader_val = DataLoader(cifar10_val, batch_size=64,
                        sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN, 50000)))

cifar10_test = dset.CIFAR10('./deeplearning/datasets', train=False, download=True,
                            transform=transform)
loader_test = DataLoader(cifar10_test, batch_size=64)

### Training Helpers

In [ ]:
def check_valid_accuracy(loader, model):
    # print('Checking accuracy on validation set')
    if not loader.dataset.train:
        print('Checking accuracy on test set')
    num_correct = 0
    num_samples = 0
    model.eval()  # set model to evaluation mode
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)
            scores = model(x)
            _, preds = scores.max(1)
            num_correct += (preds == y).sum()
            num_samples += preds.size(0)
        acc = float(num_correct) / num_samples
        # print('Got %d / %d correct (%.2f)' % (num_correct, num_samples, 100 * acc))
    return acc

def train_model(model, optimizer, epochs=1):
    """
    Train a model on CIFAR-10 using the PyTorch Module API and prints model
    accuracies during training.

    Inputs:
    - model: A PyTorch Module giving the model to train.
    - optimizer: An Optimizer object we will use to train the model
    - epochs: (Optional) A Python integer giving the number of epochs to train for

    Returns: Lists of validation accuracies at the end of each epoch.
    """
    model = model.to(device=device)  # move the model parameters to CPU/GPU
    train_accs = []
    val_accs = []
    for e in range(epochs):
        for t, (x, y) in enumerate(loader_train):
            model.train()  # put model to training mode
            x = x.to(device=device, dtype=dtype)  # move to device, e.g. GPU
            y = y.to(device=device, dtype=torch.long)

            scores = model(x)
            loss = F.cross_entropy(scores, y)

            # Zero out all of the gradients for the variables which the optimizer
            # will update.
            optimizer.zero_grad()

            # This is the backwards pass: compute the gradient of the loss with
            # respect to each trainable parameter of the model.
            loss.backward()

            # Actually update the parameters of the model using the gradients
            # computed by the backwards pass.
            optimizer.step()

            if t % print_every == 0:
                acc = check_valid_accuracy(loader_val, model)
                print(f"Epoch [{e + 1}/{epochs}]: Iter {t}, loss = {round(loss.item(), 4)}, validation accuracy = {100*acc}%")
        val_accs.append(check_valid_accuracy(loader_val, model))
    return val_accs

### Define 3 Layer MLP and CNN

In [ ]:
class ThreeLayerConvNet(nn.Module):
    def __init__(self, in_channel, channel_1, channel_2, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channel, channel_1, 5, stride=1, padding=2)
        self.conv2 = nn.Conv2d(channel_1, channel_2, 3, stride=1, padding=1)
        self.classifier = nn.Linear(channel_2 * 32 * 32, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = x.flatten(start_dim=1)
        x = self.classifier(x)

        return x

class ThreeLayerMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = self.fc1(x)
        x=  F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.classifier(x)

        return x

### Count the parameters

Count all weight and bias parameters. With the settings below, the MLP has slightly more total parameters than the CNN. The helper can also count only parameters with `requires_grad=True`, which will be useful when we freeze the CNN backbone.


In [ ]:
def count_parameters(model, trainable_only=False):
    return sum(
        p.numel() for p in model.parameters()
        if not trainable_only or p.requires_grad
    )

input_size = 3 * 32 * 32
in_channel = 3
channel_1 = 100
channel_2 = 100
num_classes = 10
hidden_size = 350

mlp_model = ThreeLayerMLP(input_size, hidden_size, num_classes).to(device)
cnn_model = ThreeLayerConvNet(in_channel, channel_1, channel_2, num_classes).to(device)

num_params_mlp = count_parameters(mlp_model)
num_params_cnn = count_parameters(cnn_model)

print(f"# of params in MLP model : {num_params_mlp}")
print(f"# of params in CNN model : {num_params_cnn}")


### Train the models

Run the next cell to train the MLP and CNN for the same number of epochs. Runtime depends on your hardware; training on a CPU can take much longer. If time is limited, set `total_epochs = 2` and use that same value for the frozen-backbone experiment below.


In [ ]:
learning_rate = 1e-2

mlp_optimizer = optim.SGD(mlp_model.parameters(), lr=learning_rate)
cnn_optimizer = optim.SGD(cnn_model.parameters(), lr=learning_rate)

# total_epochs = 2
total_epochs = 5

print("Start MLP training...")
mlp_accuracy = train_model(mlp_model, mlp_optimizer, total_epochs)
print("Start CNN training...")
cnn_accuracy = train_model(cnn_model, cnn_optimizer, total_epochs)
print(f"Final MLP validation accuracy: {100 * mlp_accuracy[-1]:.1f}%")
print(f"Final CNN validation accuracy: {100 * cnn_accuracy[-1]:.1f}%")


**Q.** What are the final validation accuracies of the MLP and CNN in your run? Which model performs better, and how might their inductive biases help explain the result?

**A.** 

**Q.** Suppose we initialize a new CNN, freeze its convolutional layers at their random initial values, and train only its final linear classifier. Which parameters receive gradients and updates? How could you implement this?

**A.** 


The following experiment creates a **new, randomly initialized CNN** and freezes its convolutional backbone before any training. It does not reuse the CNN trained above. Only its final linear classifier is optimized.

The convolutional parameters remain part of the model, so freezing them reduces the number of trainable parameters but not the total number of stored parameters. Run the following cells and compare validation accuracy, total parameters, and trainable parameters with the previous models.


In [ ]:
class ThreeLayerConvNetDetached(nn.Module):
    def __init__(self, in_channel, channel_1, channel_2, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channel, channel_1, 5, stride=1, padding=2)
        self.conv2 = nn.Conv2d(channel_1, channel_2, 3, stride=1, padding=1)
        self.classifier = nn.Linear(channel_2 * 32 * 32, num_classes)

        # Freeze the randomly initialized convolutional weights and biases.
        self.conv1.requires_grad_(False)
        self.conv2.requires_grad_(False)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = x.flatten(start_dim=1).detach()
        x = self.classifier(x)

        return x

In [ ]:
detached_cnn_model = ThreeLayerConvNetDetached(
    in_channel, channel_1, channel_2, num_classes
).to(device)
detached_cnn_optimizer = optim.SGD(
    (p for p in detached_cnn_model.parameters() if p.requires_grad),
    lr=learning_rate,
)
print(f"Frozen CNN total parameters: {count_parameters(detached_cnn_model):,}")
print(f"Frozen CNN trainable parameters: {count_parameters(detached_cnn_model, trainable_only=True):,}")


In [ ]:
print("Start Detached CNN training...")
detached_cnn_accuracy = train_model(detached_cnn_model, detached_cnn_optimizer, total_epochs)
print(f"Final frozen-CNN validation accuracy: {100 * detached_cnn_accuracy[-1]:.1f}%")


**Q.** Compare the frozen-backbone model with the fully trained CNN and MLP. What happened to validation accuracy, the total parameter count, and the number of trainable parameters? Explain why a frozen random backbone can still support learning without guaranteeing a particular accuracy.

**A.** 

**Q.** How many trainable parameters, including biases, does the frozen-backbone model contain? What fraction of the full CNN parameter count is this?

**A.** 
